# UI Element Detector — Kaggle training

Two stages, same split and seed:

1. **YOLO26n** finds element boxes on the full screenshot.
2. **DenseNet121, Xception41, SE-ResNet50, PyramidNet-110, Residual Attention Network** classify each detected crop.

## Before you press Run

1. **Upload the data once:** kaggle.com → *Datasets* → *New Dataset* → drag `ui-data.zip` → title `ui-data` → *Create*. Do **not** extract it.
2. Open this notebook → **File → Import notebook** (or *New Notebook* and paste the cells).
3. Right panel → **Session options**:
   * **Accelerator: GPU T4 x2** (or *GPU P100* if offered — faster)
   * **Internet: On** (needed for `git clone` and the ImageNet weights)
4. **Add Input** → *Your datasets* → `ui-data`.

Phone-verify your account first, otherwise the GPU and internet switches stay locked.

Session limits: **12 h per session, 30 h GPU per week**. This training is ~12–15 h, so plan on 2–3 sessions.


In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Code

Cloned from GitHub, so it is always the current version. (If you run without internet, upload `ui-element-detector-code.zip` as a dataset instead and unzip it to `/kaggle/working/project`.)

In [ ]:
import os
REPO_URL = 'https://github.com/AnaraHayat/ui-Element-detector.git'

%cd /kaggle/working
!rm -rf project
!git clone -q $REPO_URL project
%cd /kaggle/working/project
!pip install -q timm ultralytics
print('code ready:', os.getcwd())

## 2. Extract the dataset

The zip is read from `/kaggle/input/...` (read-only, never modified). It is unpacked into `/kaggle/working/project/data`, which is where every script expects it.

The next cell also removes the absolute `path:` line that older `data.yaml` files carry — without that, Ultralytics looks for `D:/python/...` and fails.

In [ ]:
import glob, os, pathlib, zipfile, yaml

zips = glob.glob('/kaggle/input/*/ui-data.zip') + glob.glob('/kaggle/input/*/*/ui-data.zip')
assert zips, 'ui-data.zip not found under /kaggle/input - add it via Add Input + Your datasets'
ZIP_PATH = zips[0]
print('zip:', ZIP_PATH, round(os.path.getsize(ZIP_PATH) / 1e9, 2), 'GB')

if not pathlib.Path('data/yolo/data.yaml').exists():
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall('data')
print('extracted')

# make data.yaml portable: relative to this file, no machine-specific path
p = pathlib.Path('data/yolo/data.yaml')
cfg = yaml.safe_load(p.read_text())
cfg.pop('path', None)
p.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(p.read_text())
print('train jpgs:', len(list(pathlib.Path('data/yolo/images/train').glob('*.jpg'))))
print('val   jpgs:', len(list(pathlib.Path('data/yolo/images/val').glob('*.jpg'))))
print('test  jpgs:', len(list(pathlib.Path('data/yolo/images/test').glob('*.jpg'))))

## 3. Dataset sanity check

In [ ]:
from ultralytics.data.utils import check_det_dataset
d = check_det_dataset('data/yolo/data.yaml')
print('train:', d['train'], '| nc:', d['nc'])
print(list(d['names'].values()))

## 4. Smoke test (a few minutes)

Downloads the three ImageNet checkpoints and trains densenet121 for 1 epoch on 500 crops. If this passes, the GPU, the data and the weights are all fine. **Do not skip it.**

In [ ]:
!python scripts/models/check_models.py --size 128
!python scripts/training/train_cnn.py --model densenet121 --epochs 1 --max-crops 500 --val-per-class 20 --batch-size 32 --out-suffix _smoke

## 5. Five classifiers

Each run is skipped if its checkpoint is already saved in `/kaggle/working/saved`, so after a session reset you only train what is missing.

To split the work over sessions, edit `MODELS` — e.g. `MODELS = ['densenet121', 'xception41']` in session 1.

In [ ]:
import os, shutil
MODELS = ['densenet121', 'xception41', 'seresnet50', 'pyramidnet110', 'residual_attention']
SAVE = '/kaggle/working/saved'
os.makedirs(SAVE, exist_ok=True)

for m in MODELS:
    if os.path.exists(f'{SAVE}/models_{m}_best.pt'):
        print('already trained, skipping:', m)
        continue
    print('=' * 70); print('TRAINING', m)
    !python scripts/training/train_cnn.py --model {m}
    shutil.copy(f'results/{m}/best.pt', f'{SAVE}/models_{m}_best.pt')

## 6. Detector (YOLO26n)

In [ ]:
import os, shutil
if os.path.exists('/kaggle/working/saved/models_yolo26n_best.pt'):
    print('already trained, skipping: yolo26n')
else:
    !python scripts/training/train_yolo.py --model yolo26n
    shutil.copy('results/yolo26n/weights/best.pt', '/kaggle/working/saved/models_yolo26n_best.pt')

## 7. Package the results

Everything below `/kaggle/working` survives if you **Save Version** (top right → *Save & Run All* or *Quick Save*). Afterwards open the notebook's **Output** tab and download `ui-results.zip`.

In [ ]:
import shutil, os
shutil.make_archive('/kaggle/working/ui-results', 'zip', 'results')
print('saved:', os.path.getsize('/kaggle/working/ui-results.zip') / 1e6, 'MB')
for f in sorted(os.listdir('/kaggle/working/saved')):
    print(' ', f)